# ResNet18/ResNet50 Random Search: Train/Val Accuracy


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, MultipleLocator, PercentFormatter
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
})

# Volle Textbreite, wie in moe/plots/big_randomsearch_plots_final.ipynb.
FIGSIZE_PAIR = (6.8, 2.8)
MAX_EPOCH = None


In [ ]:
ROOT = Path(
    "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/"
    "tensorboard_runs/tensorboard_runs_random_search"
).resolve()

PLOTS_DIR = Path(
    "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/plot"
).resolve()

OUT_DIR = PLOTS_DIR / "figures_flarge_randomsearch"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ARCHITECTURES = {
    "ResNet18": "resnet18",
    "ResNet50": "resnet50",
}

PDF_PATH = OUT_DIR / "flarge_randomsearch_resnet18_resnet50_train_val_accuracy.pdf"
ROOT, OUT_DIR, PDF_PATH


In [ ]:
def parse_hyperparameters(run_path: Path) -> dict:
    container = run_path.parent.name
    run_name = run_path.name

    dropout_match = re.search(r"dropout([0-9.]+)", container)
    lr_match = re.search(r"lr([0-9.eE+-]+)", run_name)
    wd_match = re.search(r"wd([0-9.eE+-]+)", run_name)
    container_match = re.search(r"Container(\d+)", container)

    return {
        "container": container,
        "run_dir": run_name,
        "dropout": float(dropout_match.group(1)) if dropout_match else np.nan,
        "lr": float(lr_match.group(1)) if lr_match else np.nan,
        "weight_decay": float(wd_match.group(1)) if wd_match else np.nan,
        "container_id": int(container_match.group(1)) if container_match else -1,
    }


def read_scalar_dir(path: Path, preferred_tag: str) -> list[dict]:
    if not path.exists():
        return []

    event_files = sorted(path.glob("events.out.tfevents.*"))
    if not event_files:
        return []

    accumulator = EventAccumulator(str(path), size_guidance={"scalars": 0})
    accumulator.Reload()
    tags = accumulator.Tags().get("scalars", [])
    if not tags:
        return []

    tag = preferred_tag if preferred_tag in tags else tags[0]
    return [
        {
            "step": int(event.step),
            "value": float(event.value),
            "wall_time": float(event.wall_time),
        }
        for event in accumulator.Scalars(tag)
    ]


def run_has_test_accuracy(run_path: Path) -> bool:
    try:
        accumulator = EventAccumulator(str(run_path), size_guidance={"scalars": 0})
        accumulator.Reload()
        return "Accuracy/test" in accumulator.Tags().get("scalars", [])
    except Exception:
        return False


def collect_accuracy_scalars(root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    scalar_rows = []
    run_rows = []

    for architecture_label, architecture_key in ARCHITECTURES.items():
        candidate_runs = sorted(
            run_dir
            for container in root.glob(f"*{architecture_key}*")
            if container.is_dir()
            for run_dir in container.glob("lr*_wd*")
            if run_dir.is_dir()
        )

        finished_runs = [run_dir for run_dir in candidate_runs if run_has_test_accuracy(run_dir)]

        for run_index, run_dir in enumerate(finished_runs, start=1):
            meta = parse_hyperparameters(run_dir)
            run_id = f"{architecture_key}_{run_index:02d}"
            config_key = run_id

            run_rows.append({
                "architecture": architecture_label,
                "architecture_key": architecture_key,
                "run_id": run_id,
                "config_key": config_key,
                "run_path": str(run_dir),
                **meta,
            })

            for split, subdir, tag in [
                ("train", "Accuracy_train", "Accuracy"),
                ("val", "Accuracy_val", "Accuracy"),
            ]:
                values = read_scalar_dir(run_dir / subdir, tag)
                if not values:
                    warnings.warn(f"Keine Accuracy-Werte fuer {run_dir / subdir}")
                    continue
                for row in values:
                    scalar_rows.append({
                        "architecture": architecture_label,
                        "architecture_key": architecture_key,
                        "run_id": run_id,
                        "config_key": config_key,
                        "split": split,
                        **row,
                    })

    return pd.DataFrame(scalar_rows), pd.DataFrame(run_rows)


In [ ]:
scalars, runs = collect_accuracy_scalars(ROOT)

summary = (
    runs.groupby("architecture", sort=False)
    .agg(runs=("run_id", "nunique"))
    .reindex(ARCHITECTURES.keys())
)

display(summary)
display(runs[["architecture", "run_id", "dropout", "lr", "weight_decay", "container"]].head())


In [ ]:
def tag_frame(architecture: str, split: str) -> pd.DataFrame:
    frame = scalars.loc[
        (scalars["architecture"] == architecture)
        & (scalars["split"] == split)
    ].copy()

    if MAX_EPOCH is not None:
        frame = frame.loc[frame["step"] <= MAX_EPOCH]

    return frame


CONFIG_ORDER_BY_ARCH = {
    architecture: runs.loc[runs["architecture"] == architecture, "config_key"].tolist()
    for architecture in ARCHITECTURES.keys()
}

CONFIG_COLORS_BY_ARCH = {}
for architecture, config_order in CONFIG_ORDER_BY_ARCH.items():
    colors = plt.colormaps["tab20"].resampled(max(len(config_order), 1))
    CONFIG_COLORS_BY_ARCH[architecture] = {
        key: colors(i)
        for i, key in enumerate(config_order)
    }


def style_axis(ax, ylabel=None, ylim=None, ytick_step=None, percent=False):
    ax.set_xlabel(
        "Epoche",
        labelpad=2,
    )

    if ylabel is not None:
        ax.set_ylabel(
            ylabel,
            labelpad=2,
        )

    if ylim is not None:
        ax.set_ylim(
            *ylim,
        )

    if ytick_step is not None:
        ax.yaxis.set_major_locator(
            MultipleLocator(
                ytick_step,
            )
        )

    if percent:
        ax.yaxis.set_major_formatter(
            PercentFormatter(
                xmax=1.0,
                decimals=0,
            )
        )

    ax.xaxis.set_major_locator(
        MaxNLocator(
            nbins=6,
            integer=True,
        )
    )

    ax.grid(
        axis="both",
        color="0.90",
        linewidth=0.7,
        linestyle="-",
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def set_epoch_xlim(axes):
    max_step = scalars["step"].max()
    right = MAX_EPOCH if MAX_EPOCH is not None else max_step
    if pd.notna(right):
        right = right + max(1, right * 0.02)
        for ax in np.ravel(axes):
            ax.set_xlim(
                0,
                right,
            )


def speichere_abbildung(fig, path: Path, extra_artists=()):
    bbox_extra_artists = tuple(fig.legends) + tuple(extra_artists)
    fig.savefig(
        path,
        bbox_inches="tight",
        bbox_extra_artists=bbox_extra_artists,
        pad_inches=0.08,
    )
    print(path)


In [ ]:
def plot_resnet_train_val_pair():
    fig, axes = plt.subplots(
        1,
        2,
        figsize=FIGSIZE_PAIR,
        sharex=True,
        sharey=True,
    )

    for ax, architecture in zip(
        axes,
        ARCHITECTURES.keys(),
    ):
        train_frame = tag_frame(
            architecture,
            "train",
        )

        val_frame = tag_frame(
            architecture,
            "val",
        )

        for config_key in CONFIG_ORDER_BY_ARCH[architecture]:
            color = CONFIG_COLORS_BY_ARCH[architecture][config_key]

            run_train = train_frame.loc[
                train_frame["config_key"] == config_key
            ].sort_values(
                "step",
            )

            run_val = val_frame.loc[
                val_frame["config_key"] == config_key
            ].sort_values(
                "step",
            )

            if not run_train.empty:
                ax.plot(
                    run_train["step"],
                    run_train["value"],
                    color=color,
                    linestyle="--",
                    linewidth=0.8,
                    alpha=0.65,
                )

            if not run_val.empty:
                ax.plot(
                    run_val["step"],
                    run_val["value"],
                    color=color,
                    linestyle="-",
                    linewidth=0.9,
                    alpha=0.85,
                )

        ax.set_title(
            architecture,
            pad=3,
        )

        style_axis(
            ax,
            ylabel="Genauigkeit" if ax is axes[0] else None,
            ylim=(0.50, 1.00),
            ytick_step=0.10,
            percent=True,
        )

    set_epoch_xlim(
        axes,
    )

    style_handles = [
        Line2D(
            [0], [0],
            color="0.25",
            linestyle="--",
            linewidth=0.9,
            label="Training",
        ),
        Line2D(
            [0], [0],
            color="0.25",
            linestyle="-",
            linewidth=0.9,
            label="Validierung",
        ),
    ]

    axes[0].legend(
        handles=style_handles,
        loc="lower right",
        frameon=True,
        framealpha=0.9,
        borderpad=0.3,
        fontsize=6.2,
        handlelength=1.5,
        labelspacing=0.2,
    )

    fig.subplots_adjust(
        left=0.075,
        right=0.98,
        top=0.91,
        bottom=0.20,
        wspace=0.18,
    )

    speichere_abbildung(
        fig,
        PDF_PATH,
    )

    plt.show()
    return fig, axes


plot_resnet_train_val_pair()
